## This program uses a fine-tuned DeepSeek model to detect flaws in Python and Mamba code and produce refactored versions of the code.

In [ ]:
# ----------------------------------------------------------------------------
# 
# Please put your program in Test1.txt for Mamba or Test1.py for Python.
# The result will be saved in Results.txt

# Full_Model=True     for using a full model
# Full_Model=False    for using LoRA adapter

# choose 1 for version 1.
# choose 1 for Python or 2 for Mamba in "Language" variable.
# ----------------------------------------------------------------------------
Language=1
if Language==1:
   InputFile="Test1.py"
else:
   InputFile="Test1.txt"
    
OutputFile = "Results.txt"
# ----------------------------------------------------------------------------
Full_Model= True #False
Version=1 


In [ ]:
import time
import torch
import os
import re
import ast
import random
import numpy as np
import warnings

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from huggingface_hub import login

warnings.filterwarnings("ignore")

In [ ]:
DEVICE="cuda:0" if torch.cuda.is_available() else "cpu"

In [ ]:
token = "YOUR_TOKEN_HERE" 

login(token=token)

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
# ----------------------------------------------------------------------------
if Full_Model==False:
   if Language==1: 
      CheckPoint='HA-Siala/RefactoringPy-DeepSeek-v0.1'  
   else:
      CheckPoint='HA-Siala/Mamba-DeepSeek-v0.1'
       
   deepseek_checkpoint = "deepseek-ai/deepseek-coder-6.7b-base"
   tokenizer = AutoTokenizer.from_pretrained(CheckPoint, trust_remote_code=True)

   if tokenizer.pad_token is None:
      tokenizer.pad_token = tokenizer.eos_token

   base_model = AutoModelForCausalLM.from_pretrained(
      deepseek_checkpoint,
      torch_dtype=torch.bfloat16,
      device_map="auto",
      trust_remote_code=True
   )

   model = PeftModel.from_pretrained(base_model, CheckPoint)
   model.eval()   
else:
   if Language==1: 
      CheckPoint='HA-Siala/RefactoringPy-DeepSeek-full-v0.1'       
   else:
      CheckPoint='HA-Siala/Mamba-DeepSeek-full-v0.1'
    
   model = AutoModelForCausalLM.from_pretrained(CheckPoint, torch_dtype=torch.bfloat16, device_map="auto", low_cpu_mem_usage=True) 
   tokenizer = AutoTokenizer.from_pretrained(CheckPoint, use_fast=True)
   if tokenizer.pad_token is None:
      tokenizer.pad_token = tokenizer.eos_token
model.eval() 
# ----------------------------------------------------------------------------

In [ ]:
# ----------------------------------------------------------------------------
def format_time(seconds):
   h = int(seconds // 3600)
   m = int((seconds % 3600) // 60)
   s = seconds % 60
   result = ""
   if h > 0:
      result += f"{h}h "
   if m > 0 or h > 0:  
      result += f"{m}m "
   result += f"{s:.6f}s"
   return result
# ----------------------------------------------------------------------------   
def CleanText(text):
   if not text:
      return ""
   text = str(text)
   text = text.split("###")[0]
   return text
# ----------------------------------------------------------------------------   

In [ ]:
# ----------------------------------------------------------------------------
def AnalyzeCode(code):
   if Language==1: 
      Instruction="""Analyze the following Python code, detect any flaws, with a short explanation, and give one or more corrected versions of the code. Ensure that none of the refactored options repeat or reproduce the original code; only the improved version should be shown."""       
   else:    
      Instruction="""Analyze the following Mamba code, detect any flaws, with a short explanation, and give one or more corrected versions of the code. Ensure that none of the refactored options repeat or reproduce the original code; only the improved version should be shown."""
   prompt = f"""You are an AI programming assistant, utilizing the DeepSeek Coder model.
### Instruction:
{Instruction}

### Input:
{code}

### Response:
"""
   inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
   InputTokens = inputs["input_ids"].shape[1] 
   with torch.no_grad():
      outputs = model.generate(
         **inputs,
         max_new_tokens= 1024, #8000, #32768, 
         do_sample=False,
         num_beams=1,
         temperature=0.0,
         top_p=1.0,
         repetition_penalty=1.1,
         pad_token_id=tokenizer.pad_token_id,
         eos_token_id=tokenizer.eos_token_id,
      )
   response = tokenizer.decode(
      outputs[0][inputs["input_ids"].shape[1]:],
      skip_special_tokens=True
   )
   return response, InputTokens
# ----------------------------------------------------------------------------
def PrintResult(raw_output, out_file):
   out_file.write("Flaws:\n")
   if not raw_output:
      out_file.write("INVALID OUTPUT\n\n")
      return
   try:
      text = CleanText(str(raw_output))
      flaws = re.findall(r'"Flaw"\s*:\s*"([^"]+)"', text)
      explanations = re.findall(r'"Explanation"\s*:\s*"([^"]+)"', text)
      for i in range(min(len(flaws), len(explanations))):
         out_file.write(f"   - {flaws[i]} : {explanations[i]}\n")
      out_file.write("\nRefactored versions code:\n\n")
      refactored = re.search(
         r'"Refactored Versions"\s*:\s*"([\s\S]+)"',
         text
      )
      if refactored:
         cleaned_code = refactored.group(1)
         cleaned_code = cleaned_code.replace('\\"', '"')  
         cleaned_code = cleaned_code.replace("\\n", "\n")
         out_file.write(cleaned_code)
      else:
         out_file.write("INVALID OUTPUT")
      out_file.write("\n\n")
   except Exception:
      out_file.write("INVALID OUTPUT\n\n")
# ----------------------------------------------------------------------------
def ReadFile():
   if not os.path.exists(InputFile):
      print(f"Error: File '{InputFile}' does not exist.")
      return None
   if (Language==1 and not InputFile.endswith('.py')) or (Language==2 and not InputFile.endswith('.txt')):
      print(f"Warning: '{InputFile}' may not be a right file.")
   with open(InputFile, 'r', encoding='utf-8') as file:
      content = file.read()
      return content
# ----------------------------------------------------------------------------
content = ReadFile()
if content:
   print("Successfully read file!")
   print() 
   StartTime=time.time()
   result, InputTokens = AnalyzeCode(content)
   EndTime=time.time()
   InferenceTime=EndTime - StartTime
   TimePerInputToken = InferenceTime / InputTokens
   with open(OutputFile, "w", encoding="utf-8") as f:
      PrintResult(result, f)
      print(f"Saved to: {OutputFile}")      
   print(f"Inference Time: {format_time(InferenceTime)}")
   print(f"Execution time per input token: {format_time(TimePerInputToken)}")
else:
   print("Error in reading the input file.") 
# ----------------------------------------------------------------------------
